# SETUP

In [1]:
BUCKET_NAME = "phiri-rhema-lab3"
ML1M_FILES = ["ratings.dat", "movies.dat", "users.dat", "README"]
DATA_DIRECTORY = "dataset/moviedata/ml-1m"

In [9]:
!pip install faiss-cpu==1.7.4

  Obtaining dependency information for faiss-cpu==1.7.4 from https://files.pythonhosted.org/packages/61/e6/3b740163e7fcce1220417ef6c18a7bc00b9d11b64264f3abf483a3771153/faiss_cpu-1.7.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 64.0 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: faiss-cpu
    Found existing installation: faiss-cpu 1.13.2
    Uninstalling faiss-cpu-1.13.2:
      Successfully uninstalled faiss-cpu-1.13.2


In [10]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss
import boto3

/home/ec2-user/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# TASK 1

In [50]:
# check if dataset exists on S3 bucket 

def s3_prefix_exists(bucket_name, prefix):
    s3 = boto3.client('s3')

    for fname in ML1M_FILES:
            key = f"{prefix}/{fname}"
            response = s3.list_objects_v2(Bucket=bucket_name, Prefix=key, MaxKeys=1)
            if "Contents" not in response:
                print(f"  Missing: {key}")
                return False

    print("All files already in S3.")
    return True

In [51]:
#from datarec.datasets import Movielens

import urllib.request
import zipfile

def download_movielens_1m(data_dir="dataset/moviedata"):
    if os.path.exists(data_dir):
        print("Dataset already exists locally. Skipping download.")
        return data_dir

    os.makedirs(data_dir, exist_ok=True)

    url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
    zip_path = os.path.join(data_dir, "ml-1m.zip")

    urllib.request.urlretrieve(url, zip_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(data_dir)

    os.remove(zip_path)  # clean up the zip
    print(f"Dataset saved to {data_dir}")
    return data_dir

In [52]:
from botocore.exceptions import NoCredentialsError

def upload_to_s3(local_dir, bucket, prefix):
    s3 = boto3.client('s3')
    
    for fname in ML1M_FILES:
        local_file = os.path.join(local_dir, fname)
        s3_key = f"{prefix}/{fname}"
        try:
            s3.upload_file(local_file, bucket, s3_key)
            print(f"  Uploaded: {s3_key}")
        except FileNotFoundError:
            print(f"  File not found: {local_file}")
            return False
        except NoCredentialsError:
            print("Credentials not available")
            return False
    
    print("All files uploaded successfully.")
    return True

In [53]:
def task1_pipeline():
    
    folder_exists = s3_prefix_exists(BUCKET_NAME, 'ML-1M_dataset')
    
    # in bucket
    if folder_exists:
        return
    
    # not in bucket
    else:
        download_movielens_1m(data_dir="dataset/moviedata")
        upload_to_s3('dataset/moviedata/ml-1m', BUCKET_NAME, 'ML-1M_dataset')

    return

In [54]:
task1_pipeline()

All files already in S3.


# TASK 2

In [55]:
cols = ['ID', 'Title', 'Genres']

def load_movies_before_1980(data_dir):
    movies_path = f"{data_dir}/movies.dat"
    
    movies = pd.read_csv(movies_path, sep="::",encoding='latin1', names=cols, engine="python")
    movies["Year"] = movies["Title"].str.extract(r'\((\d{4})\)').astype(int)
    movies["Genres"] = movies["Genres"].str.replace("|", ", ", regex=False)
    
    filtered_movies = movies[movies["Year"] <= 1980].reset_index(drop=True)
    return filtered_movies

In [56]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [57]:
# BERT encoder helper
@torch.no_grad() # no back propagation
def bert_embed(texts, max_len=128):
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)


In [58]:
def generate_bert_embeddings(movies):
    movies["text"] = movies["Title"] + ". " + movies["Genres"]
    movie_vecs = bert_embed(movies["text"].tolist())
    movie_ids = movies["ID"].tolist()


    # Build ANN index (inner product works with normalized vectors)
    index = faiss.IndexFlatIP(movie_vecs.shape[1])
    index.add(movie_vecs)
    np.save("movie_embeddings.npy", movie_vecs)
    np.save("movie_ids.npy", np.array(movie_ids))
    
    
    return movie_ids, movie_vecs, index


In [59]:
def save_and_upload_embeddings(movie_ids, movie_vecs, movies, bucket=BUCKET_NAME):

    s3 = boto3.client('s3')
    response = s3.list_objects_v2(Bucket=bucket, Prefix="embeddings/", MaxKeys=1)
    if "Contents" in response:
        print("Embeddings already in S3. Skipping upload.")
        return

  
    embedding_df = pd.DataFrame(movie_vecs, columns=[f"emb_{i}" for i in range(movie_vecs.shape[1])])
    meta_df = movies[["ID", "Title", "Year"]].reset_index(drop=True)
    result_df = pd.concat([meta_df, embedding_df], axis=1)
    result_df.to_csv("movie_embeddings.csv", index=False)


    s3.upload_file("movie_embeddings.csv", bucket, "embeddings/movie_embeddings.csv")
    np.save("movie_ids.npy", np.array(movie_ids))
    s3.upload_file("movie_ids.npy", bucket, "embeddings/movie_ids.npy")

In [60]:
def task2_pipeline():
       
    movies = load_movies_before_1980(DATA_DIRECTORY)
    movie_ids, movie_vecs, index = generate_bert_embeddings(movies)

    print(f"Number of movies: {len(movie_ids)}")
    print(f"Embedding shape: {movie_vecs.shape}")
    
    save_and_upload_embeddings(movie_ids, movie_vecs, movies)
    
    return

In [61]:
task2_pipeline()

Number of movies: 887
Embedding shape: (887, 768)
Embeddings already in S3. Skipping upload.


# TASK 3

In [62]:
cols_ratings = ['UserID', 'MovieID', 'Rating','Timestamp']

def load_ratings(data_dir):
    ratings_path = f"{data_dir}/ratings.dat"
    
    ratings = pd.read_csv(ratings_path, sep="::",encoding='latin1', names=cols_ratings, engine="python")
    
    return ratings

In [63]:
cols_users = ['UserID','Gender','Age','Occupation','ZipCode']

def load_users(data_dir):
    users_path = f"{data_dir}/users.dat"
    
    users = pd.read_csv(users_path, sep="::",encoding='latin1', names=cols_users, engine="python")
    
    return users

In [64]:
def load_embeddings():
    movie_vecs = np.load("movie_embeddings.npy")
    movie_ids = np.load("movie_ids.npy")
    return movie_ids, movie_vecs

In [65]:
def build_user_text(user_id, ratings, movies, N=3):
    
    hist = (
        ratings[ratings["UserID"] == user_id]
        .sort_values("Timestamp")
        .tail(N)["MovieID"]
        .tolist()
    )
    if not hist: # cold user
        return "no history", set()
    
    movies_indexed = movies.set_index("ID")
    text = movies_indexed.loc[
        [mid for mid in hist if mid in movies_indexed.index], "text"
    ].tolist()
    return " ".join(text), set(hist)

In [66]:
def build_faiss_index(movie_vecs):
    index = faiss.IndexFlatIP(movie_vecs.shape[1])
    index.add(movie_vecs)
    return index

In [67]:
def recommend(user_id, ratings, movies, movie_ids, movie_vecs, index, k=5):
    user_text, seen = build_user_text(user_id, ratings, movies)

    if user_text == "no history":
        popularity = (
            ratings[ratings["MovieID"].isin(movie_ids)]
            .groupby("MovieID").size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
        )
        top_ids = popularity["MovieID"].head(k).tolist()
        return movies[movies["ID"].isin(top_ids)][["ID", "Title", "Year", "Genres"]]

   
    u = bert_embed([user_text])                   
    scores, idx = index.search(u, k + len(seen))  

    recs = []
    for j in idx[0]:
        iid = int(movie_ids[j])  
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break

    rec_df = movies[movies["ID"].isin(recs)][["ID", "Title", "Year", "Genres"]]
    rec_df = rec_df.set_index("ID").loc[recs].reset_index()
    return rec_df

In [68]:
def get_top_user(ratings, users):
    user_counts = ratings.groupby("UserID").size().reset_index(name="RatingCount")
    threshold = user_counts["RatingCount"].quantile(0.95)
    top_users = user_counts[user_counts["RatingCount"] >= threshold]

    selected_user_id = top_users.sample(1, random_state=42)["UserID"].values[0]
    user_info = users[users["UserID"] == selected_user_id].iloc[0]
    user_ratings = ratings[ratings["UserID"] == selected_user_id]

    return user_info, user_ratings


In [69]:
def save_recs(records, bucket=BUCKET_NAME, local_path="recommendations.csv", s3_key="recommendations/recommendations.csv"):
    rows = []
    for record in records:
        base = {k: v for k, v in record.items() if k != "Recommended_Movies"}
        for i, movie in enumerate(record["Recommended_Movies"], 1):
            row = base.copy()
            row[f"Rec_{i}_ID"] = movie["ID"]
            row[f"Rec_{i}_Title"] = movie["Title"]
            row[f"Rec_{i}_Year"] = movie["Year"]
            row[f"Rec_{i}_Genres"] = movie["Genres"]
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(local_path, index=False)
    s3 = boto3.client("s3")
    s3.upload_file(local_path, bucket, s3_key)
    print(f"  Uploaded to s3")

In [70]:
cols = ['ID', 'Title', 'Genres']
def load_all_movies(data_dir):
    movies_path = f"{data_dir}/movies.dat"
    
    movies = pd.read_csv(movies_path, sep="::",encoding='latin1', names=cols, engine="python")
    movies["Year"] = movies["Title"].str.extract(r'\((\d{4})\)').astype(int)
    movies["Genres"] = movies["Genres"].str.replace("|", ", ", regex=False)
    
    #filtered_movies = movies[movies["Year"] <= 1980].reset_index(drop=True)
    return movies

In [74]:
def task_pipeline(data_dir="dataset/moviedata/ml-1m", pre1980_only=True):

    ratings = load_ratings(data_dir)
    users = load_users(data_dir)
    movies = load_movies_before_1980(data_dir) if pre1980_only else load_all_movies(data_dir)
    movies["text"] = movies["Title"] + ". " + movies["Genres"]

    
    movie_vecs = bert_embed_batched(movies["text"].tolist())
    movie_ids = movies["ID"].to_numpy()

 
    index = build_faiss_index(movie_vecs)

    
    cold_recs = recommend("cold_user", ratings, movies, movie_ids, movie_vecs, index)
    cold_record = {
        "User_Type": "Cold",
        "Last_Interaction_Time": "N/A",
        "N_Interactions": 0,
        "Avg_Rating": "N/A",
        "UserID": "N/A",
        "Gender": "N/A",
        "Age": "N/A",
        "Occupation": "N/A",
        "Recommended_Movies": cold_recs[["ID", "Title", "Year", "Genres"]].to_dict(orient="records")
    }
    print(f"  Cold recs: {cold_recs['Title'].tolist()}")

    
    user_info, user_ratings = get_top_user(ratings, users)
    top_recs = recommend(user_info["UserID"], ratings, movies, movie_ids, movie_vecs, index)
    top_record = {
        "User_Type": "Top",
        "Last_Interaction_Time": pd.to_datetime(user_ratings["Timestamp"].max(), unit="s").strftime("%Y-%m-%d %H:%M:%S"),
        "N_Interactions": len(user_ratings),
        "Avg_Rating": round(user_ratings["Rating"].mean(), 2),
        "UserID": int(user_info["UserID"]),
        "Gender": user_info["Gender"],
        "Age": int(user_info["Age"]),
        "Occupation": int(user_info["Occupation"]),
        "Recommended_Movies": top_recs[["ID", "Title", "Year", "Genres"]].to_dict(orient="records")
    }
    print(f"  Top recs: {top_recs['Title'].tolist()}")

    suffix = "pre1980" if pre1980_only else "full"
 
    save_recs(
        [cold_record, top_record],
        local_path=f"recommendations_{suffix}.csv",
        s3_key=f"recommendations/recommendations_{suffix}.csv"
    )


In [75]:
task_pipeline()

  Cold recs: ['Star Wars: Episode IV - A New Hope (1977)', 'Godfather, The (1972)', 'Star Wars: Episode V - The Empire Strikes Back (1980)', 'Alien (1979)', 'Airplane! (1980)']
  Top recs: ['Champ, The (1979)', 'Gods Must Be Crazy, The (1980)', 'Long Goodbye, The (1973)', 'Idolmaker, The (1980)', 'Great Santini, The (1979)']
  Uploaded to s3


# Task 4

In [77]:
def bert_embed_batched(texts, batch_size=32):
    all_vecs = []
    total = len(texts)
    for i in range(0, total, batch_size):
        batch = texts[i:i + batch_size]
        vecs = bert_embed(batch)
        all_vecs.append(vecs)
    return np.vstack(all_vecs)

In [78]:
task_pipeline(pre1980_only=False)

  Cold recs: ['Star Wars: Episode IV - A New Hope (1977)', 'Jurassic Park (1993)', 'Star Wars: Episode V - The Empire Strikes Back (1980)', 'Star Wars: Episode VI - Return of the Jedi (1983)', 'American Beauty (1999)']
  Top recs: ['Celtic Pride (1996)', 'American Tail: Fievel Goes West, An (1991)', 'Grosse Pointe Blank (1997)', "April Fool's Day (1986)", 'Carpool (1996)']
  Uploaded to s3


# TASK 5

In [86]:
my_ratings = [
    {"MovieID": 28, "Title": "Persuasion(1995)", "Rating": 4},
    {"MovieID": 3554,  "Title": "Love and Basketball (2000)",   "Rating": 2},
    {"MovieID": 39,  "Title": "Clueless(1995)",   "Rating": 4},
    {"MovieID": 919,  "Title": "Wizard of Oz, The (1939)",   "Rating": 1},
    {"MovieID": 145,  "Title": "Bad Boys(1995)",   "Rating": 3},
    {"MovieID": 3114,  "Title": " Toy Story 2 (1999)",   "Rating": 5},
    {"MovieID": 356,  "Title": "Forrest Gump (1994)",   "Rating": 4},
    {"MovieID": 2628,  "Title": "Star Wars: Episode I - The Phantom Menace (1999)",   "Rating": 1},
    {"MovieID": 1580,  "Title": "Men in Black (1997)",   "Rating": 3},
    {"MovieID": 480,  "Title": "Jurassic Park (1993)",   "Rating": 4}   
]

In [87]:
def build_my_profile(my_ratings, movies_full, movie_ids_full, movie_vecs_full):
    my_df = pd.DataFrame(my_ratings)
    my_movie_ids = my_df["MovieID"].tolist()
    my_movie_ratings = my_df["Rating"].tolist()
    
    indices = [np.where(movie_ids_full == mid)[0][0] for mid in my_movie_ids if mid in movie_ids_full]
    weights = np.array([r for mid, r in zip(my_movie_ids, my_movie_ratings) if mid in movie_ids_full], dtype="float32")
    
    vecs = movie_vecs_full[indices]
    weighted = (vecs * weights[:, None]).sum(axis=0)
    profile = weighted / np.linalg.norm(weighted)
    return profile, set(my_movie_ids)

In [88]:
def recommend_for_me(my_ratings, movies_full, movie_ids_full, movie_vecs_full, index_full, k=5):
    
    profile, seen = build_my_profile(my_ratings, movies_full,movie_ids_full, movie_vecs_full)
    
    scores, idx = index_full.search(profile.reshape(1, -1), k + len(seen))
    
    recs = []
    for j in idx[0]:
        iid = int(movie_ids_full[j])
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break
    
    return movies_full[movies_full["ID"].isin(recs)][["ID", "Title", "Year", "Genres"]]

In [89]:
def save_profile_and_recs(my_ratings, recs, bucket=BUCKET_NAME):

    profile_df = pd.DataFrame(my_ratings)
    profile_df.to_csv("my_profile.csv", index=False)
    
    recs.to_csv("my_recommendations.csv", index=False)
    
    s3 = boto3.client("s3")
    s3.upload_file("my_profile.csv", bucket, "my_profile/my_profile.csv")
    s3.upload_file("my_recommendations.csv", bucket, "my_profile/my_recommendations.csv")

In [94]:
def task5_pipeline():
    movie_vecs_full = np.load("movie_embeddings_full.npy")
    movie_ids_full = np.load("movie_ids_full.npy")
    movies_full = load_all_movies("dataset/moviedata/ml-1m")
    index_full = build_faiss_index(movie_vecs_full)

    
    profile, seen = build_my_profile(my_ratings, movies_full, movie_ids_full, movie_vecs_full)
    
    my_recs = recommend_for_me(my_ratings, movies_full, movie_ids_full, movie_vecs_full, index_full)
    
    save_profile_and_recs(my_ratings, my_recs)
    print("Saved to s3")
    print(my_recs)


In [95]:
task5_pipeline()

Saved to s3
        ID                  Title  Year                     Genres
120    122       Boomerang (1992)  1992            Comedy, Romance
683    692            Solo (1996)  1996   Action, Sci-Fi, Thriller
1110  1126  Drop Dead Fred (1991)  1991            Comedy, Fantasy
2316  2385      Home Fries (1998)  1998            Comedy, Romance
2654  2723     Mystery Men (1999)  1999  Action, Adventure, Comedy
